# Lab03 — Build Artifacts & Deploy to Hugging Face Spaces
Chạy từng cell theo thứ tự. Chỉ cần chạy 1 lần là build xong và deploy luôn.

## Cell 1 — Cấu hình (SỬA Ở ĐÂY TRƯỚC KHI CHẠY)

In [ ]:
# ============================================================
# SỬA 2 GIÁ TRỊ NÀY TRƯỚC KHI CHẠY
# ============================================================
HF_TOKEN   = "YOUR_HF_TOKEN_HERE"   # Paste token mới vào đây — ĐỪNG commit token thật lên GitHub
HF_REPO_ID = "dangvanky/lab03-22127227"  # username/space-name

# Tham số build (có thể giữ nguyên)
MAX_RECORDS = 1500
GNN_EPOCHS  = 60
EVAL_LIMIT  = 200

GITHUB_REPO = "https://github.com/dvk26/Lab03.git"
REPO_DIR    = "/content/lab03"
# ============================================================

## Cell 2 — Clone repo từ GitHub

In [2]:
import os

os.system(f"rm -rf {REPO_DIR}")
os.system(f"git clone {GITHUB_REPO} {REPO_DIR}")
os.chdir(REPO_DIR)
print("Commit:", os.popen("git rev-parse --short HEAD").read().strip())

Commit: a3c0bf9


## Cell 3 — Cài thư viện cần thiết để build (không cần llama-cpp)

In [3]:
%%capture
# Chỉ cài những gì cần cho build_all.py — bỏ qua llama-cpp và gradio
!pip install -q \
    datasets \
    huggingface-hub \
    llama-index-core \
    networkx \
    numpy \
    pandas \
    scikit-learn \
    "sentence-transformers>=3.0.0" \
    torch \
    torch-geometric \
    tqdm

!pip install -q -e .   # cài lab03 package

print("Cài xong!")

## Cell 4 — Build artifacts + Train GNN

In [4]:
import os

os.environ["MAX_RECORDS"] = str(MAX_RECORDS)
os.environ["GNN_EPOCHS"]  = str(GNN_EPOCHS)
os.environ["EVAL_LIMIT"]  = str(EVAL_LIMIT)

# Build graph + embeddings + train GNN (tất cả trong 1 lệnh)
!python scripts/build_all.py

README.md: 100% 436/436 [00:00<00:00, 1.20MB/s]
data/train-00000-of-00001.parquet: 100% 2.21M/2.21M [00:00<00:00, 3.49MB/s]
Generating train split: 100% 15549/15549 [00:00<00:00, 206870.05 examples/s]
modules.json: 100% 349/349 [00:00<00:00, 865kB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 498kB/s]
README.md: 10.5kB [00:00, 22.1MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 205kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.57MB/s]
model.safetensors: 100% 90.9M/90.9M [00:00<00:00, 148MB/s]
Loading weights: 100% 103/103 [00:00<00:00, 2011.24it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.5

## Cell 5 — Kiểm tra artifacts đã có đủ chưa

In [5]:
from pathlib import Path
import json

required = [
    "artifacts/graph_nodes.json",
    "artifacts/graph_edges.json",
    "artifacts/manifest.json",
    "artifacts/raw_embeddings.npy",
    "artifacts/edge_index.npy",
    "artifacts/structural_embeddings.npy",
]

print("=== Kiểm tra artifacts ===")
all_ok = True
for f in required:
    exists = Path(f).exists()
    size   = Path(f).stat().st_size if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {f}  ({size/1024:.1f} KB)")
    if not exists:
        all_ok = False

print()
if all_ok:
    print("Tất cả artifacts đã có — sẵn sàng deploy!")
else:
    print("THIẾU FILE — build_all.py chưa chạy xong hoặc bị lỗi.")

# In metrics nếu có
metrics_path = Path("evaluation/retrieval_metrics.json")
if metrics_path.exists():
    print("\n=== Retrieval Metrics ===")
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))

=== Kiểm tra artifacts ===
  [OK] artifacts/graph_nodes.json  (2808.7 KB)
  [OK] artifacts/graph_edges.json  (3581.7 KB)
  [OK] artifacts/manifest.json  (44.4 KB)
  [OK] artifacts/raw_embeddings.npy  (9981.1 KB)
  [OK] artifacts/edge_index.npy  (890.4 KB)
  [OK] artifacts/structural_embeddings.npy  (9981.1 KB)

Tất cả artifacts đã có — sẵn sàng deploy!

=== Retrieval Metrics ===
{
  "num_queries": 200,
  "hit_rate_at_k": 0.91,
  "mrr_at_k": 0.7869166666666666,
  "by_question_type": {
    "causes": {
      "count": 23,
      "hit_rate_at_k": 0.9565217391304348,
      "mrr_at_k": 0.8804347826086957
    },
    "complications": {
      "count": 23,
      "hit_rate_at_k": 0.8260869565217391,
      "mrr_at_k": 0.6811594202898551
    },
    "diagnosis": {
      "count": 22,
      "hit_rate_at_k": 0.9545454545454546,
      "mrr_at_k": 0.7916666666666665
    },
    "overview": {
      "count": 22,
      "hit_rate_at_k": 0.9545454545454546,
      "mrr_at_k": 0.8954545454545454
    },
    "preven

## Cell 6 — Deploy lên Hugging Face Space

In [6]:
from huggingface_hub import HfApi, login

login(token=HF_TOKEN, add_to_git_credential=False)

api = HfApi()

api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="space",
    space_sdk="docker",
    exist_ok=True,
)

print(f"Đang upload lên {HF_REPO_ID} ...")
api.upload_folder(
    folder_path=REPO_DIR,
    repo_id=HF_REPO_ID,
    repo_type="space",
    ignore_patterns=[
        ".git/*",
        ".venv/*",
        "**/__pycache__/*",
        "**/*.pyc",
        "**/*.pyo",
        ".env",
        "models/*",          # model GGUF sẽ tự download lúc chạy
        "storage/*",          # LlamaIndex graph store, large, not needed at runtime
        "evaluation/*",       # metrics output, not needed at runtime
        ".pytest_cache/*",
    ],
)

print(f"\nDone! Space đang rebuild tại:")
print(f"https://huggingface.co/spaces/{HF_REPO_ID}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Đang upload lên dangvanky/lab03-22127227 ...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ifacts/raw_embeddings.npy: 100%|##########| 10.2MB / 10.2MB            

  /content/lab03/Lab03.pdf    : 100%|##########|  151kB /  151kB            

  .../artifacts/edge_index.npy: 100%|##########|  912kB /  912kB            

  ...tifacts/gnn_checkpoint.pt:   5%|5         | 61.2kB / 1.19MB            

  ...structural_embeddings.npy:   5%|5         |  525kB / 10.2MB            


Done! Space đang rebuild tại:
https://huggingface.co/spaces/dangvanky/lab03-22127227
